## Normal distribution dashboards (all 8 axes)

This section starts again from the raw CSV so it can run on its own. The charts are Plotly, so they zoom, pan and show values on hover, and clicking a legend entry hides or shows it. Each dashboard has its own filter panel and an **Apply filter** button. Timestamps in the file are UTC.

### Imports

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

### Loading the data
All eight axes and the timestamp. The empty Axis #9 to #14 columns and the Trait column are left out.

In [ ]:
axes_cols = [f'Axis #{i}' for i in range(1, 9)]

robot_df = pd.read_csv('robot_data.csv', usecols=axes_cols + ['Time'])
print(robot_df.shape)
robot_df.head()

### Share of zero readings
About 65% of every axis is zero, so a distribution of all readings is one tall spike at 0. The dashboards default to non-zero readings for that reason.

In [ ]:
zero_share = (robot_df[axes_cols] == 0).mean().mul(100).round(1)
print(zero_share)
print('\nRows where every axis is zero:', round((robot_df[axes_cols].sum(axis=1) == 0).mean() * 100, 1), '%')

### Timestamp feature engineering
Time is parsed to a datetime, then broken into date, hour, elapsed minutes and the gap since the previous reading. Day part and elapsed quarter give the portions used for filtering, and an active row is any row where at least one axis is above zero.

In [ ]:
robot_df['Time'] = pd.to_datetime(robot_df['Time'], utc=True)
robot_df = robot_df.sort_values('Time').reset_index(drop=True)

robot_df['Date'] = robot_df['Time'].dt.strftime('%Y-%m-%d')
robot_df['Hour'] = robot_df['Time'].dt.hour
robot_df['Elapsed Min'] = (robot_df['Time'] - robot_df['Time'].iloc[0]).dt.total_seconds() / 60
robot_df['Time Gap'] = robot_df['Time'].diff().dt.total_seconds().fillna(0)

robot_df['Day Part'] = pd.cut(
    robot_df['Hour'],
    bins=[-1, 5, 11, 17, 23],
    labels=['Night (00-05)', 'Morning (06-11)', 'Afternoon (12-17)', 'Evening (18-23)']
)
robot_df['Elapsed Quarter'] = pd.qcut(robot_df['Elapsed Min'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
robot_df['Active Row'] = robot_df[axes_cols].sum(axis=1) > 0

for col in ['Day Part', 'Elapsed Quarter']:
    robot_df[col] = robot_df[col].astype(str)

print('Readings more than 60 seconds after the previous one:', (robot_df['Time Gap'] > 60).sum())
print(robot_df.groupby('Day Part')['Active Row'].agg(['size', 'mean']).round(3))
robot_df[['Time', 'Date', 'Hour', 'Elapsed Min', 'Time Gap', 'Day Part', 'Elapsed Quarter', 'Active Row']].head()

### Portions, filters and plot helpers
Portions are listed in the order they were recorded. The filters are portion, hour range, which readings to keep and an optional time average (the mean of the readings inside each 1, 5 or 15 minute window, which pulls the shape much closer to a bell). The helper builds the histogram, the fitted normal curve, the median and the chosen percentile lines. Log scale uses log(1 + x) and keeps tick labels in the original units.

In [ ]:
INK, MUTED, GRID, SURFACE = '#0b0b0b', '#52514e', '#e6e5e1', '#fcfcfb'
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']
FONT = 'Inter, Segoe UI, Helvetica, Arial, sans-serif'
NICE = np.array([0, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100, 200])

portion_options = {'All data': None}
for col in ['Day Part', 'Elapsed Quarter', 'Date']:
    for val in pd.unique(robot_df[col]):
        portion_options[f'{col}: {val}'] = (col, val)

window_options = {'Each reading': None, '1 min average': 1, '5 min average': 5, '15 min average': 15}


def portion_rows(portion, hours):
    d = robot_df
    key = portion_options[portion]
    if key:
        d = d[d[key[0]] == key[1]]
    return d[d['Hour'].between(*hours)]


def pick_values(d, axis, readings, window):
    if readings == 'Non-zero readings':
        d = d[d[axis] > 0]
    elif readings == 'Active rows':
        d = d[d['Active Row']]
    minutes = window_options[window]
    if minutes:
        return d.groupby(d['Elapsed Min'] // minutes)[axis].mean().to_numpy()
    return d[axis].to_numpy()


def norm_pdf(x, mu, sd):
    return np.exp(-0.5 * ((x - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))


def rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return f'rgba({r},{g},{b},{alpha})'


def axis_style(title=None, **kw):
    style = dict(gridcolor=GRID, zeroline=False, showline=False, ticks='', **kw)
    if title:
        style['title'] = dict(text=title)
    return style


def log_axis(lo, hi):
    pos = np.log1p(NICE)
    keep = (pos >= lo) & (pos <= hi)
    return dict(tickmode='array', tickvals=pos[keep].tolist(), ticktext=[f'{v:g}' for v in NICE[keep]])


def base_layout(title, subtitle=None, height=580):
    text = f'<b>{title}</b>'
    if subtitle:
        text += f'<br><span style="font-size:13px;color:{MUTED}">{subtitle}</span>'
    return dict(
        template='plotly_white', height=height, paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
        font=dict(family=FONT, size=14, color=INK),
        title=dict(text=text, x=0.01, xanchor='left'),
        margin=dict(l=65, r=30, t=100, b=80),
        hovermode='closest', hoverlabel=dict(bgcolor='white', font=dict(size=13, family=FONT)),
        legend=dict(orientation='h', x=0, y=-0.16),
    )


def nd_parts(values, scale, pcts, hist_color, curve_color, k=1, legend=True, labels=True):
    log = scale == 'Log'
    x = np.log1p(values) if log else values
    mu, sd = x.mean(), x.std()

    dens, edges = np.histogram(x, bins=50, density=True)
    mid = (edges[:-1] + edges[1:]) / 2
    edges_orig = np.expm1(edges) if log else edges
    lo, hi = min(x.min(), mu - 3.5 * sd), max(x.max(), mu + 3.5 * sd)
    grid = np.linspace(lo, hi, 400)
    pdf = norm_pdf(grid, mu, sd)
    ymax = max(dens.max(), pdf.max()) * 1.15

    ref = '' if k == 1 else str(k)
    ax = dict(xaxis='x' + ref, yaxis='y' + ref)

    traces = [
        dict(type='bar', x=mid, y=dens, width=np.diff(edges),
             marker=dict(color=rgba(hist_color, 0.45), line=dict(width=0)),
             customdata=np.column_stack([edges_orig[:-1], edges_orig[1:]]).tolist(),
             hovertemplate='%{customdata[0]:.2f} to %{customdata[1]:.2f}<br>density %{y:.3f}<extra>Readings</extra>',
             name='Readings', legendgroup='bars', showlegend=legend, **ax),
        dict(type='scatter', mode='lines', x=grid, y=pdf, line=dict(color=curve_color, width=3.5),
             customdata=np.expm1(grid) if log else grid,
             hovertemplate='%{customdata:.2f}<br>normal density %{y:.3f}<extra>Normal fit</extra>',
             name='Normal fit', legendgroup='fit', showlegend=legend, **ax),
    ]

    levels = sorted(set(pcts) | {50})
    q = np.percentile(values, levels)
    qx = np.log1p(q) if log else q

    if 25 in levels and 75 in levels:
        a, b = qx[levels.index(25)], qx[levels.index(75)]
        traces.append(dict(type='scatter', mode='lines', x=[a, b, b, a, a], y=[0, 0, ymax, ymax, 0],
                           fill='toself', fillcolor=rgba(MUTED, 0.10), line=dict(width=0), hoverinfo='skip',
                           name='P25 to P75', legendgroup='iqr', showlegend=legend, **ax))

    notes, first_pct = [], True
    for i, (lv, val, pos) in enumerate(zip(levels, q, qx)):
        is_med = lv == 50
        name = 'Median' if is_med else f'P{lv}'
        traces.append(dict(type='scatter', mode='lines', x=[float(pos)] * 30, y=np.linspace(0, ymax, 30),
                           line=dict(color=INK if is_med else MUTED, width=3.5 if is_med else 1.8,
                                     dash='solid' if is_med else 'dash'),
                           hovertemplate=f'{name}: {val:.2f}<extra></extra>',
                           name='Median' if is_med else 'Percentiles',
                           legendgroup='median' if is_med else 'pct',
                           showlegend=legend and (is_med or first_pct), **ax))
        if not is_med:
            first_pct = False
        if labels:
            notes.append(dict(x=float(pos), y=ymax, xref='x' + ref, yref='y' + ref, text=f'{name} {val:.2f}',
                              showarrow=False, xanchor='left', yanchor='top', yshift=-18 * (i % 3),
                              font=dict(size=11, color=INK), bgcolor=rgba(SURFACE, 0.85)))

    info = dict(n=len(values), mean=float(values.mean()), sd=float(values.std()),
                median=float(np.median(values)),
                skew=float(np.mean(((values - values.mean()) / values.std()) ** 3)),
                lo=float(lo), hi=float(hi), ymax=float(ymax))
    return traces, notes, info

### Filter panel
Each dashboard gets its own panel. **Apply filter** redraws the chart with the current choices and **Reset filters** goes back to the defaults.

In [ ]:
def build_controls(with_axis=False, with_portion=True, with_compare=False):
    c = {}
    if with_axis:
        c['axis'] = widgets.Dropdown(options=axes_cols, value='Axis #2', description='Axis')
    if with_portion:
        c['portion'] = widgets.Dropdown(options=list(portion_options), description='Portion')
    if with_compare:
        c['by'] = widgets.Dropdown(options=['Day Part', 'Elapsed Quarter', 'Date'], description='Compare by')
    c['readings'] = widgets.ToggleButtons(options=['Non-zero readings', 'Active rows', 'All readings'], description='Readings')
    c['window'] = widgets.Dropdown(options=list(window_options), description='Average')
    c['scale'] = widgets.ToggleButtons(options=['Raw', 'Log'], description='Scale')
    c['hours'] = widgets.IntRangeSlider(value=[0, 23], min=0, max=23, description='Hours (UTC)')
    c['pcts'] = widgets.SelectMultiple(options=[1, 5, 10, 25, 75, 90, 95, 99], value=(5, 25, 75, 95), description='Percentiles', rows=4)
    c['apply'] = widgets.Button(description='Apply filter', button_style='primary')
    c['reset'] = widgets.Button(description='Reset filters')
    return c


def launch(controls, render):
    out = widgets.Output()
    names = [k for k in controls if k not in ('apply', 'reset')]
    defaults = {k: controls[k].value for k in names}

    def run(_=None):
        with out:
            clear_output(wait=True)
            render(**{k: controls[k].value for k in names})

    def reset(_):
        for k, v in defaults.items():
            controls[k].value = v
        run()

    controls['apply'].on_click(run)
    controls['reset'].on_click(reset)
    display(widgets.VBox([controls[k] for k in names] + [widgets.HBox([controls['apply'], controls['reset']]), out]))
    run()

### Dashboard 1: one axis
Histogram of the filtered readings with the fitted normal curve. The median is the solid black line, the selected percentiles are dashed, and the grey band is P25 to P75. Hover any bar, the curve or a line for its values. Raw readings are lumpy because the robot holds the same current for stretches, so try Log scale or a time average if the bell is not showing. The line under the title shows the zero share of the portion before zeros are dropped.

In [ ]:
def render_single(axis, portion, readings, window, scale, hours, pcts):
    d = portion_rows(portion, hours)
    vals = pick_values(d, axis, readings, window)
    if len(vals) < 15:
        print('Not enough readings for this filter')
        return
    traces, notes, info = nd_parts(vals, scale, pcts, SERIES[0], SERIES[1])
    zero_pct = (d[axis] == 0).mean() * 100
    subtitle = (f"n = {info['n']:,} | mean {info['mean']:.2f} | sd {info['sd']:.2f} | "
                f"skew {info['skew']:.2f} | zeros in portion {zero_pct:.0f}%")

    layout = base_layout(f'{axis} | {portion}', subtitle)
    xaxis = axis_style('Current' + (' (log scale)' if scale == 'Log' else ''), range=[info['lo'], info['hi']])
    if scale == 'Log':
        xaxis.update(log_axis(info['lo'], info['hi']))
    layout['xaxis'] = xaxis
    layout['yaxis'] = axis_style('Density')
    layout['annotations'] = notes
    display(go.Figure(dict(data=traces, layout=layout)))


launch(build_controls(with_axis=True), render_single)

### Dashboard 2: all 8 axes
The same view for every axis side by side, using one filter panel. The median is in each panel title and the lines mark the chosen percentiles. The legend at the bottom controls all eight panels at once.

In [ ]:
def render_grid(portion, readings, window, scale, hours, pcts):
    d = portion_rows(portion, hours)
    layout = base_layout(f'All axes | {portion}', height=780)
    layout['legend'] = dict(orientation='h', x=0, y=-0.07)
    layout['margin'] = dict(l=50, r=20, t=90, b=70)

    gap = 0.05
    width = (1 - 3 * gap) / 4
    rows = [(0.57, 0.90), (0.09, 0.42)]
    traces, notes = [], []

    for k, col in enumerate(axes_cols, start=1):
        r, c = divmod(k - 1, 4)
        x0 = c * (width + gap)
        y0, y1 = rows[r]
        ref = '' if k == 1 else str(k)
        xaxis = axis_style(domain=[x0, x0 + width], anchor='y' + ref)
        layout['yaxis' + ref] = axis_style(domain=[y0, y1], anchor='x' + ref)

        vals = pick_values(d, col, readings, window)
        if len(vals) < 15:
            notes.append(dict(x=x0, y=y1, xref='paper', yref='paper', text=f'<b>{col}</b>  not enough readings',
                              showarrow=False, xanchor='left', yanchor='bottom', font=dict(size=13)))
            layout['xaxis' + ref] = xaxis
            continue

        t, _, info = nd_parts(vals, scale, pcts, SERIES[0], SERIES[1], k=k, legend=(k == 1), labels=False)
        traces += t
        xaxis['range'] = [info['lo'], info['hi']]
        if scale == 'Log':
            xaxis.update(log_axis(info['lo'], info['hi']))
        layout['xaxis' + ref] = xaxis
        notes.append(dict(x=x0, y=y1, xref='paper', yref='paper',
                          text=f"<b>{col}</b>  median {info['median']:.2f}  n = {info['n']:,}",
                          showarrow=False, xanchor='left', yanchor='bottom', font=dict(size=13)))

    layout['annotations'] = notes
    display(go.Figure(dict(data=traces, layout=layout)))


launch(build_controls(), render_grid)

### Dashboard 3: compare portions
One axis, with a normal curve for each portion of the recording on the same chart. The dashed line in each colour is that portion's median, and clicking a legend entry hides that portion. The table underneath gives the percentiles for each portion, so a shift between portions shows in both the chart and the numbers.

In [ ]:
def render_compare(axis, by, readings, window, scale, hours, pcts):
    log = scale == 'Log'
    levels = sorted(set(pcts) | {50})
    groups = pd.unique(robot_df[by])[:4]

    series = []
    for g, color in zip(groups, SERIES):
        d = robot_df[(robot_df[by] == g) & robot_df['Hour'].between(*hours)]
        vals = pick_values(d, axis, readings, window)
        if len(vals) >= 15:
            series.append((g, color, vals, np.log1p(vals) if log else vals))
    if not series:
        print('Not enough readings for this filter')
        return

    lo = min(min(x.min(), x.mean() - 3 * x.std()) for _, _, _, x in series)
    hi = max(np.percentile(x, 99.5) for _, _, _, x in series)
    grid = np.linspace(lo, hi, 500)
    pdfs = [norm_pdf(grid, x.mean(), x.std()) for _, _, _, x in series]
    ymax = max(p.max() for p in pdfs) * 1.15

    traces, notes, rows = [], [], []
    for i, ((g, color, vals, x), pdf) in enumerate(zip(series, pdfs)):
        label = f'{g} (n = {len(vals):,})'
        traces.append(dict(type='scatter', mode='lines', x=grid, y=pdf, fill='tozeroy', fillcolor=rgba(color, 0.14),
                           line=dict(color=color, width=3.5), customdata=np.expm1(grid) if log else grid,
                           hovertemplate=g + '<br>%{customdata:.2f}<br>density %{y:.3f}<extra></extra>',
                           name=label, legendgroup=g))
        med = float(np.median(vals))
        med_x = float(np.log1p(med) if log else med)
        traces.append(dict(type='scatter', mode='lines', x=[med_x] * 30, y=np.linspace(0, ymax, 30),
                           line=dict(color=color, width=2, dash='dash'),
                           hovertemplate=f'{g} median: {med:.2f}<extra></extra>',
                           name=f'{g} median', legendgroup=g, showlegend=False))
        notes.append(dict(x=med_x, y=ymax, xref='x', yref='y', text=f'{g} median {med:.2f}', showarrow=False,
                          xanchor='left', yanchor='top', yshift=-18 * i, font=dict(size=11, color=INK),
                          bgcolor=rgba(SURFACE, 0.85)))
        row = {'Portion': g, 'n': len(vals), 'mean': vals.mean(), 'sd': vals.std()}
        for lv, val in zip(levels, np.percentile(vals, levels)):
            row['Median' if lv == 50 else f'P{lv}'] = val
        rows.append(row)

    layout = base_layout(f'{axis} | by {by}')
    xaxis = axis_style('Current' + (' (log scale)' if log else ''), range=[lo, hi])
    if log:
        xaxis.update(log_axis(lo, hi))
    layout['xaxis'] = xaxis
    layout['yaxis'] = axis_style('Density', range=[0, ymax])
    layout['annotations'] = notes
    display(go.Figure(dict(data=traces, layout=layout)))

    table = pd.DataFrame(rows).round(2)
    display(go.Figure(dict(
        data=[dict(type='table',
                   header=dict(values=list(table.columns), fill=dict(color=INK), font=dict(color='white', size=13),
                               align='left', height=34),
                   cells=dict(values=[table[c].tolist() for c in table.columns], fill=dict(color=SURFACE),
                              line=dict(color=GRID), font=dict(color=INK, size=13), align='left', height=30))],
        layout=dict(height=80 + 32 * len(table), margin=dict(l=10, r=10, t=10, b=10), paper_bgcolor=SURFACE))))


launch(build_controls(with_axis=True, with_portion=False, with_compare=True), render_compare)